# Walking and Biking Image-Location Points

This notebook displays walking and biking routes and their proposed image-location points from the included saved JSON files. Points were previously generated approximately every **200 feet**, with additional points at turns of **30 degrees or more**.

Running this notebook makes **no Google API calls** and requires **no API key**.

Click a point to view its downloaded photo and elevation. Zoom to level 16 or closer to reveal points. Photos are read from `Walking Photos` and `Biking Photos`; missing photos are labeled.


In [ ]:
# Imports and map-building setup. No API calls are made.
import json
import csv
import base64
from html import escape
from pathlib import Path

import folium
from branca.element import MacroElement, Template

# Match the neighborhood colors used in the original project.
NEIGHBORHOOD_COLORS = {
    "Airport District": "salmon",
    "Downtown SLO": "blue",
    "Foothill": "cadetblue",
    "Laguna": "purple",
    "Laguna Lake": "purple",
    "LoBro": "darkred",
    "LoHI": "green",
    "SLO South": "darkgreen",
    "Mojo": "orange",
    "RR": "brown",
}


def resolve_project_path(path):
    """Resolve relative paths from the repo root when possible."""
    candidate = Path(path)
    if candidate.is_absolute():
        return candidate
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "saved_route_points").exists():
            return (base / candidate).resolve()
    return (Path.cwd() / candidate).resolve()


def build_point_map(saved_data_path, travel_label, point_zoom=16):
    """Build a Folium map strictly from saved JSON; no API calls occur here."""
    saved_data = json.loads(resolve_project_path(saved_data_path).read_text(encoding="utf-8"))
    saved_routes = saved_data["routes"]

    point_map = folium.Map(location=[35.27, -120.66], zoom_start=13)
    folium.map.CustomPane("route-photo-points", z_index=625).add_to(point_map)
    point_map.get_root().header.add_child(folium.Element(
        "<style>.leaflet-route-photo-points-pane { display:none; }</style>"
    ))
    route_layer = folium.FeatureGroup(
        name=f"Actual {travel_label.lower()} routes", show=True
    ).add_to(point_map)
    interval_layer = folium.FeatureGroup(
        name="Regular 200-foot points", show=True
    ).add_to(point_map)
    turn_layer = folium.FeatureGroup(
        name="Turn points", show=True
    ).add_to(point_map)
    endpoint_layer = folium.FeatureGroup(
        name="Origins and destinations", show=True
    ).add_to(point_map)

    mapped_points = []
    point_count = 0

    for route in saved_routes:
        route_points = route["route_points"]
        image_locations = route["image_locations"]
        distance_meters = route["distance_meters"]
        duration = route["duration"]
        mapped_points.extend(route_points)
        duration_minutes = round(float(duration.rstrip("s")) / 60)
        route_color = NEIGHBORHOOD_COLORS.get(route["neighborhood"], "gray")
        folium.PolyLine(
            route_points,
            color=route_color,
            weight=6,
            opacity=0.9,
            tooltip=(
                f"{travel_label} to {route['destination_name']} — "
                f"{distance_meters / 1000:.2f} km, about {duration_minutes} min"
            ),
        ).add_to(route_layer)

        photo_folder = resolve_project_path("Walking Photos" if travel_label.lower() == "walking" else "Biking Photos") / route["route_id"]
        image_rows = {}
        metadata_path = resolve_project_path(photo_folder) / "point_images.csv"
        if metadata_path.exists():
            with metadata_path.open(newline="", encoding="utf-8-sig") as metadata_file:
                image_rows = {
                    (round(float(row["latitude"]), 6), round(float(row["longitude"]), 6)): row
                    for row in csv.DictReader(metadata_file)
                }

        for location in image_locations:
            point_count += 1
            point = [location["latitude"], location["longitude"]]
            reason = location["selection_reason"]

            if "turn" in reason:
                color, radius, layer = "orange", 4, turn_layer
            elif "origin" in reason:
                color, radius, layer = "red", 5, endpoint_layer
            elif "destination" in reason:
                color, radius, layer = "black", 5, endpoint_layer
            else:
                color, radius, layer = "cadetblue", 3, interval_layer

            elevation = location.get("elevation_meters")
            resolution = location.get("elevation_resolution_meters")
            if elevation is None:
                elevation_label = "Elevation unavailable"
                elevation_details = elevation_label
            else:
                elevation_feet = elevation * 3.28084
                elevation_label = (
                    f"Elevation: {elevation:.1f} m ({elevation_feet:.0f} ft)"
                )
                elevation_details = elevation_label
                if resolution is not None:
                    elevation_details += f"<br>Resolution: {resolution:.1f} m"

            tooltip = (
                f"{route['route_id']} to {route['destination_name']} — "
                f"point {location['location_index']}, {reason} — "
                f"{elevation_label}"
            )
            popup_html = (
                f"<b>{route['route_id']} to {route['destination_name']}</b><br>"
                f"Point {location['location_index']} ({reason})<br>"
                f"{elevation_details}"
            )
            metadata = image_rows.get((round(point[0], 6), round(point[1], 6)), {})
            filename = metadata.get("image_filename", "")
            image_path = resolve_project_path(photo_folder / filename)
            if filename and image_path.is_file():
                encoded = base64.b64encode(image_path.read_bytes()).decode("ascii")
                popup_html += (
                    f'<br><img src="data:image/jpeg;base64,{encoded}" '
                    'alt="Street View at this point" style="display:block;width:auto;max-width:min(560px,65vw);height:auto;max-height:45vh;object-fit:contain">'
                )
            else:
                popup_html += "<br>No downloaded image available for this point."
            folium.CircleMarker(
                point,
                radius=radius,
                pane="route-photo-points",
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.8,
                tooltip=tooltip,
                popup=folium.Popup(popup_html, max_width=600, max_height="60vh", lazy=True, keep_in_view=True, auto_pan_padding_top_left=[20, 20], auto_pan_padding_bottom_right=[20, 20]),
            ).add_to(layer)

    if mapped_points:
        point_map.fit_bounds(mapped_points)
    folium.LayerControl().add_to(point_map)
    print(
        f"Mapped {point_count} image locations from "
        f"{len(saved_routes)} saved routes. No API calls were made."
    )
    visibility = MacroElement()
    visibility._template = Template("""
    {% macro script(this, kwargs) %}
    (function () {
        var map = {{ this.map_name }};
        function updatePoints() {
            map.getPane('route-photo-points').style.display =
                map.getZoom() >= {{ this.point_zoom }} ? 'block' : 'none';
        }
        map.on('popupopen', function(event) {
            var popup = event.popup;
            var element = popup.getElement();
            if (!element) return;
            element.querySelectorAll('img').forEach(function(image) {
                var fitPopup = function() {
                    image.style.maxHeight = Math.max(80, map.getSize().y - 180) + 'px';
                    image.style.maxWidth = Math.max(100, Math.min(560, map.getSize().x - 100)) + 'px';
                    popup.update();
                };
                image.addEventListener('load', fitPopup, {once:true});
                fitPopup();
            });
        });
        map.on('zoomend', updatePoints);
        map.whenReady(updatePoints);
    })();
    {% endmacro %}
    """)
    visibility.map_name = point_map.get_name()
    visibility.point_zoom = int(point_zoom)
    visibility.add_to(point_map)
    return point_map


## 1. Map saved walking points


In [5]:
# This cell only reads the saved JSON file; it never calls Google.
walking_points_map = build_point_map(
    saved_data_path=Path("saved_route_points/walking_routes.json"),
    travel_label="Walking",
)
walking_points_map

Mapped 873 image locations from 27 saved routes. No API calls were made.


## 2. Map saved biking points


In [6]:
# This cell only reads the saved JSON file; it never calls Google.
biking_points_map = build_point_map(
    saved_data_path=Path("saved_route_points/biking_routes.json"),
    travel_label="Biking",
)
biking_points_map

Mapped 1795 image locations from 27 saved routes. No API calls were made.
